==============================================================================
ESG RAG 인덱스 구축 스크립트 (Google Colab용) — v4

v3 -> v4 수정 사항 (검증 CSV 리뷰 피드백 반영):
  1. [Cell 5c] section_title 키워드 매칭이 순서 없이 "짧은 줄에 키워드 포함"만
     보다가, 연결재무제표 주석의 "1. 회사의 개요"(각주 제목)를 대분류로 다시
     잡아버려 p.57~283이 전부 "회사의 개요"로 오염되는 버그 확인.
     -> detect_section_title이 (제목, 순서인덱스)를 반환하도록 변경하고,
        process_document에서 idx가 역행하면 무시하도록 수정 (섹션은 항상
        문서 순서대로만 진행된다고 가정).
  2. [Cell 6] DART 뷰어 푸터("전자공시시스템 dart.fss.or.kr Page N") 및
     목차 점선(".....") 청크가 그대로 인덱싱되던 문제 확인 (검증 CSV에서
     50자 미만 청크 134개, 그중 다수가 푸터).
     -> strip_noise_lines()로 페이지 텍스트에서 푸터 줄 제거,
        is_toc_chunk()로 목차성 청크는 애초에 append하지 않도록 필터 추가.
        20자 미만 잔여 청크도 함께 스킵.
  3. [Cell 5] 지속가능경영보고서는 선(border) 없는 디자인 표가 많아
     find_tables() 기본 전략(lines)으로 표를 거의 못 잡음 (검증 CSV 기준
     사업/지배구조 대비 페이지당 표 개수가 1/3 이하, 평균 길이도 짧음).
     -> extract_page_content에 doc_type 인자를 추가, sustainability_report이고
        기본 전략으로 표가 0개면 vertical/horizontal_strategy="text"로 재시도.
        호출부([Cell 6], [Cell Q], [Cell Q2]/preview 함수) 전부 doc_type 전달.

v2 -> v3 수정 사항 (GPT 피드백 반영):
  1. Chroma add_documents에 chunk_id를 명시적 ID로 전달 -> 재실행 시 중복 적재 방지
  2. 체크포인트를 {doc_id: {hash, indexed_at}} 구조로 변경 -> PDF가 바뀌면
     자동 감지, 기존 doc_id 청크를 삭제하고 재인덱싱
  3. 큰 표를 token 기준으로 행 단위 분할 (헤더 반복) -> 800토큰 룰이 표에도 적용
  4. 표 셀 클리닝 (줄바꿈/파이프 이스케이프) -> 마크다운 표 깨짐 방지
  5. indexed_at을 문서 단위로 1회만 생성 -> 청크마다 값이 미세하게 달라지는 것 방지
  [가장 중요] 6. BM25가 메타데이터 필터를 못 받던 구조적 문제 해결:
     전체 코퍼스로 BM25를 한 번에 만드는 대신, 검색 시점에 필터(company_code 등)로
     좁혀진 부분집합에서만 BM25를 만드는 FilteredHybridRetriever로 교체.
     -> 이 변경으로 기존 "BM25 전체 재구축 비용" 문제도 자연히 해소되어
        전역 BM25 pickle 저장/로드 로직 자체를 제거함 (Cell 8, Cell R 단순화)

보류한 피드백 (아직 미반영, TODO로 유지):
  - applicable_item_codes 사전 태깅 (현재 빈 배열)
  - esg_area 규칙 기반 태깅 (현재 빈 문자열)
  - 하이브리드 결합 가중치(dense:BM25) 튜닝 — 평가셋 확인 후 결정
  - 리랭커(Cross-Encoder) 적용 여부

확정된 하이퍼파라미터/모델 (팀 확인 완료):
  - 임베딩 모델: Google Gemini gemini-embedding-001 (Google AI Studio API)
  - 벡터스토어: Chroma (로컬, Google Drive에 영속 저장)
  - 청크 크기: 800 tokens / 오버랩 100 tokens (주의: cl100k_base 기준 근사치.
    한국어는 Gemini 실제 토크나이저 기준 청크당 약 450~550토큰 수준일 가능성 높음)
  - BM25 토크나이저: Kiwi (한국어 형태소 분석)
==============================================================================


### [Cell 1] 설치

In [1]:
!pip install -q langchain langchain-google-genai langchain-chroma langchain-community \
    chromadb pdfplumber kiwipiepy rank_bm25 tiktoken tenacity

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 393.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━

### [Cell 2] Google Drive 마운트 + API 키

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import getpass
import os

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google AI Studio API Key 입력: ")

Mounted at /content/drive
Google AI Studio API Key 입력: ··········


### [Cell 3] 설정값 (CONFIG)

In [3]:
CONFIG = {
    # --- 확정 ---
    "embedding_model": "gemini-embedding-001",  # Google AI Studio (Gemini Developer API)
    "chunk_size_tokens": 800,
    "chunk_overlap_tokens": 100,
    "vectorstore_backend": "chroma",
    "bm25_tokenizer": "kiwi",
    "retrieval_k": 10,  # dense/BM25 각각 몇 개씩 뽑을지

    # --- 임베딩 API 429(RESOURCE_EXHAUSTED) 대응 ---
    # 확인된 한도: 분당 3,000 요청 / 분당 100만 토큰 / 일일 무제한 (Gemini Embedding)
    # -> 배치를 키우고 딜레이를 줄임. 재시도는 순간 버스트 대비용으로 유지.
    "embed_batch_size": 100,      # 문서당 수백 개 청크를 100개씩 나눠 전송
    "embed_batch_delay_sec": 1.0,  # 배치 사이 짧은 대기 (여전히 안전 마진 목적)
    "embed_retry_max_attempts": 6,  # 429 발생 시 재시도 횟수 (지수 백오프)

    # --- 경로 ---
    "reports_root": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/보고서 3종",
    "persist_dir": "/content/drive/MyDrive/ESG프로젝트(개인)/데탐단/인덱스/3. RAG 인덱스/chroma_db",
    "manifest_csv": "/content/drive/MyDrive/ESG프로젝트(개인)/데탐단/인덱스/3. RAG 인덱스/manifest.csv",
    "checkpoint_log": "/content/drive/MyDrive/ESG프로젝트(개인)/데탐단/인덱스/3. RAG 인덱스/checkpoint_completed.json",

    # --- TODO: 평가셋 확인 후 결정 ---
    "hybrid_weights": [0.5, 0.5],  # [dense, bm25] 임시값. 튜닝 전까지 변경 금지
    "use_reranker": False,
}

# ==============================================================================
# manifest.csv는 이제 손으로 만들 필요 없음 — 아래 [Cell 4b]가 reports_root
# 폴더를 스캔해서 자동 생성함. 폴더/파일명 규칙:
#   보고서 3종/회사명(기업코드)/기업코드_연도_business.pdf   (business/governance/sustainability)
# 이 규칙을 벗어나는 폴더·파일은 자동으로 건너뛰고 콘솔에 경고 출력됨.
# ==============================================================================

### [Cell 4] 라이브러리 import

In [4]:
import hashlib
import json
import time
from datetime import datetime

import pandas as pd
import pdfplumber
import tiktoken
from kiwipiepy import Kiwi
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_message

kiwi = Kiwi()
RRF_K = 60  # Reciprocal Rank Fusion 표준 상수 (튜닝 대상 아님)

try:
    _enc = tiktoken.encoding_for_model(CONFIG["embedding_model"])
except KeyError:
    _enc = tiktoken.get_encoding("cl100k_base")


def token_len(text: str) -> int:
    """임베딩 모델 기준 토큰 수 근사 계산 (Gemini 전용 tokenizer가 없어 cl100k_base로 근사, 청크 크기 산정용 참고치)."""
    return len(_enc.encode(text))


def kiwi_tokenize(text: str):
    """BM25용 한국어 형태소 분석 토크나이저 (조사 분리 목적)."""
    return [t.form for t in kiwi.tokenize(text)]


splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size_tokens"],
    chunk_overlap=CONFIG["chunk_overlap_tokens"],
    length_function=token_len,
    separators=["\n\n", "\n", ". ", " ", ""],
)


@retry(
    wait=wait_exponential(multiplier=2, min=5, max=90),
    stop=stop_after_attempt(CONFIG["embed_retry_max_attempts"]),
    retry=retry_if_exception_message(match=".*RESOURCE_EXHAUSTED.*|.*429.*"),
    reraise=True,
)
def _add_batch_with_retry(vectorstore, batch_chunks):
    ids = [c.metadata["chunk_id"] for c in batch_chunks]
    vectorstore.add_documents(batch_chunks, ids=ids)


def add_documents_throttled(vectorstore, chunks, batch_size, delay_sec):
    """청크를 작은 배치로 나눠서 넣고, 배치 사이에 대기 -> 429(RESOURCE_EXHAUSTED) 완화.
    배치 하나가 재시도까지 다 실패하면 그 배치만 실패 처리하고 나머지는 계속 진행."""
    total = len(chunks)
    for i in range(0, total, batch_size):
        batch = chunks[i:i + batch_size]
        try:
            _add_batch_with_retry(vectorstore, batch)
        except Exception as e:
            print(f"  배치 실패 (청크 {i}~{i + len(batch)}/{total}): {e}")
            raise  # 문서 단위 실패로 취급 -> Cell 7의 except에서 처리
        if i + batch_size < total:
            time.sleep(delay_sec)

/tmp/ipykernel_7945/1235179789.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


### [Cell 4b] manifest.csv 자동 생성 (폴더 스캔)

In [5]:
# 폴더/파일명이 규칙적이라 손으로 100줄 안 쳐도 됨.
#   기대 구조: {reports_root}/회사명(기업코드)/기업코드_연도_business.pdf
#   doc_type 파일명 표기(business/governance/sustainability)를 내부 스키마
#   값(business_report 등)으로 매핑. 규칙에 안 맞는 폴더·파일은 건너뛰고 경고만 출력.
import re
from pathlib import Path

DOC_TYPE_MAP = {
    "business": "business_report",
    "governance": "governance_report",
    "sustainability": "sustainability_report",
}
FOLDER_PATTERN = re.compile(r"^(.*)\((\d{6})\)$")
FILE_PATTERN = re.compile(r"^(\d{6})_(\d{4})_(business|governance|sustainability)\.pdf$")


def build_manifest(reports_root: str) -> pd.DataFrame:
    root = Path(reports_root)
    if not root.exists():
        raise FileNotFoundError(f"경로를 찾을 수 없음: {reports_root} (Drive 마운트/폴더명 확인)")

    rows = []
    for company_dir in sorted(root.iterdir()):
        if not company_dir.is_dir():
            continue
        m = FOLDER_PATTERN.match(company_dir.name)
        if not m:
            print(f"[건너뜀] 폴더명 규칙 불일치: {company_dir.name}")
            continue
        company_name, company_code = m.group(1), m.group(2)

        for pdf_file in sorted(company_dir.glob("*.pdf")):
            fm = FILE_PATTERN.match(pdf_file.name)
            if not fm:
                print(f"[건너뜀] 파일명 규칙 불일치: {pdf_file.name}")
                continue
            file_code, year, doc_type_short = fm.groups()
            if file_code != company_code:
                print(f"[경고] 폴더코드({company_code})와 파일코드({file_code}) 불일치: {pdf_file}")
                continue
            rows.append({
                "company_code": company_code,
                "company_name": company_name,
                "year": int(year),
                "doc_type": DOC_TYPE_MAP[doc_type_short],
                "file_path": str(pdf_file),
            })

    return pd.DataFrame(rows)


manifest_df = build_manifest(CONFIG["reports_root"])

# --- 5개사 테스트용: 필요하면 아래 주석 해제해서 특정 기업코드만 남기기 ---
TEST_COMPANY_CODES = ["000660"]  # 지금은 SK하이닉스만
manifest_df = manifest_df[
    (manifest_df["company_code"] == "000660") & (manifest_df["year"] == 2025)
]

os.makedirs(os.path.dirname(CONFIG["manifest_csv"]), exist_ok=True)
manifest_df.to_csv(CONFIG["manifest_csv"], index=False, encoding="utf-8-sig")
print(f"manifest 생성 완료: {len(manifest_df)}건")
manifest_df

manifest 생성 완료: 3건


,company_code,company_name,year,doc_type,file_path
31,000660,SK하이닉스,2025,business_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...
32,000660,SK하이닉스,2025,governance_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...
33,000660,SK하이닉스,2025,sustainability_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...


### [Cell 4c] (대안) 내 드라이브에 안 넣고, 공유 링크의 폴더 ID로 직접 접근

In [6]:
# reports_root가 진짜 "공유 드라이브"(Shareddrives)가 아니라 그냥 "공유된 폴더"라서
# drive.mount()로 안 보일 때 이 셀을 Cell 4b 대신 사용.
# 사용법: 공유 링크 https://drive.google.com/drive/folders/<FOLDER_ID> 에서
#         <FOLDER_ID> 부분만 복사해서 아래 SHARED_FOLDER_ID에 붙여넣기.
#         (뷰어 권한만 있어도 동작함 — 내 드라이브에 추가할 필요 없음)
from google.colab import auth
import google.auth
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
import io

SHARED_FOLDER_ID = "여기에_폴더ID_붙여넣기"
LOCAL_CACHE_DIR = "/content/reports_cache"  # 코랩 로컬 디스크 (세션 종료 시 삭제됨, 재실행 시 재다운로드)

auth.authenticate_user()
_creds, _ = google.auth.default()
drive_service = gbuild("drive", "v3", credentials=_creds)


def _list_children(folder_id: str) -> list[dict]:
    items, page_token = [], None
    while True:
        resp = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token,
        ).execute()
        items.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return items


def _download_file(file_id: str, dest_path: str):
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    request = drive_service.files().get_media(fileId=file_id)
    with io.FileIO(dest_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()


def build_manifest_from_drive_api(root_folder_id: str, local_cache_dir: str) -> pd.DataFrame:
    rows = []
    company_folders = [
        f for f in _list_children(root_folder_id)
        if f["mimeType"] == "application/vnd.google-apps.folder"
    ]
    for folder in company_folders:
        m = FOLDER_PATTERN.match(folder["name"])
        if not m:
            print(f"[건너뜀] 폴더명 규칙 불일치: {folder['name']}")
            continue
        company_name, company_code = m.group(1), m.group(2)

        pdf_files = [f for f in _list_children(folder["id"]) if f["name"].lower().endswith(".pdf")]
        for pdf in pdf_files:
            fm = FILE_PATTERN.match(pdf["name"])
            if not fm:
                print(f"[건너뜀] 파일명 규칙 불일치: {pdf['name']}")
                continue
            file_code, year, doc_type_short = fm.groups()
            if file_code != company_code:
                print(f"[경고] 폴더코드({company_code})와 파일코드({file_code}) 불일치: {pdf['name']}")
                continue

            local_path = os.path.join(local_cache_dir, folder["name"], pdf["name"])
            if not os.path.exists(local_path):
                print(f"다운로드 중: {folder['name']}/{pdf['name']}")
                _download_file(pdf["id"], local_path)

            rows.append({
                "company_code": company_code,
                "company_name": company_name,
                "year": int(year),
                "doc_type": DOC_TYPE_MAP[doc_type_short],
                "file_path": local_path,
            })
    return pd.DataFrame(rows)


# 사용 예 (Cell 4b의 build_manifest() 대신 이걸로 manifest_df 생성):
# manifest_df = build_manifest_from_drive_api(SHARED_FOLDER_ID, LOCAL_CACHE_DIR)
# os.makedirs(os.path.dirname(CONFIG["manifest_csv"]), exist_ok=True)
# manifest_df.to_csv(CONFIG["manifest_csv"], index=False, encoding="utf-8-sig")
# print(f"manifest 생성 완료: {len(manifest_df)}건")
# manifest_df

KeyboardInterrupt: 

### [Cell 5] 표/텍스트 추출 유틸

In [7]:
def clean_cell(cell) -> str:
    """표 셀 안의 줄바꿈/파이프가 마크다운 표 구조를 깨는 것 방지."""
    if cell is None:
        return ""
    return str(cell).replace("\n", " ").replace("|", "\\|").strip()


def chunk_table(table_data: list[list], max_tokens: int) -> list[dict]:
    """큰 표를 token 기준으로 행 단위 분할 (각 조각에 헤더 반복).
    반환: [{"markdown": "...", "rows": [[...], ...]}, ...]
    """
    header = [clean_cell(c) for c in table_data[0]]
    data_rows = table_data[1:]

    header_md = (
        "| " + " | ".join(header) + " |\n"
        + "|" + "|".join(["---"] * len(header)) + "|"
    )

    def render(rows):
        body = "\n".join(
            "| " + " | ".join(clean_cell(c) for c in r) + " |" for r in rows
        )
        return header_md + ("\n" + body if body else "")

    chunks = []
    current_rows: list[list] = []
    for row in data_rows:
        candidate = current_rows + [row]
        if token_len(render(candidate)) > max_tokens and current_rows:
            chunks.append({"markdown": render(current_rows), "rows": current_rows})
            current_rows = [row]
        else:
            current_rows = candidate
    if current_rows:
        chunks.append({"markdown": render(current_rows), "rows": current_rows})

    return chunks or [{"markdown": header_md, "rows": []}]


# [v4 패치] doc_type 인자 추가: 지속가능경영보고서는 선(border) 없는 디자인
# 표가 많아 기본 lines 전략으로 표를 거의 못 잡는 문제 확인 (검증 CSV에서
# 사업/지배구조 대비 페이지당 표 개수 1/3 이하). 기본 전략으로 0개면
# text 전략(좌표 기반 텍스트 정렬)으로 한 번 더 시도.
def extract_page_content(pdf_page, doc_type: str = ""):
    """표 영역을 찾고, 그 바운딩박스를 제외한 텍스트만 반환 (표-텍스트 중복 방지).
    주의(GPT 지적): PDF 레이아웃에 따라 bbox가 완전히 안 잡힐 수 있음.
    실제 대상 PDF 몇 개는 아래 Cell Q로 반드시 샘플 확인할 것.
    """
    tables = pdf_page.find_tables()

    # [v4 패치] 지속가능경영보고서에서 기본(lines) 전략으로 표가 하나도
    # 안 잡히면 text 전략으로 재시도 (선 없는 디자인 표 대응)
    if doc_type == "sustainability_report" and not tables:
        tables = pdf_page.find_tables(
            table_settings={"vertical_strategy": "text", "horizontal_strategy": "text"}
        )

    table_bboxes = [t.bbox for t in tables]

    table_data_list = []
    for t in tables:
        try:
            extracted = t.extract()
        except Exception:
            continue
        if extracted and len(extracted) >= 2:
            table_data_list.append(extracted)

    def not_in_table(obj):
        x0, top, x1, bottom = obj.get("x0"), obj.get("top"), obj.get("x1"), obj.get("bottom")
        if x0 is None:
            return True
        for (tx0, ttop, tx1, tbottom) in table_bboxes:
            if x0 >= tx0 - 1 and x1 <= tx1 + 1 and top >= ttop - 1 and bottom <= tbottom + 1:
                return False
        return True

    non_table_page = pdf_page.filter(not_in_table)
    text = non_table_page.extract_text() or ""
    return text, table_data_list


def base_metadata(company_code, company_name, year, doc_type, doc_id,
                   chunk_id, page, content_type, source_file_hash,
                   doc_indexed_at, table_rows=None, section_title=""):
    """Chroma는 메타데이터로 str/int/float/bool만 허용 (None, list 불가)
    -> None은 빈 문자열로, list는 JSON 문자열로 변환."""
    return {
        "company_code": company_code,
        "company_name": company_name,
        "year": year,
        "doc_type": doc_type,
        "doc_id": doc_id,
        "chunk_id": chunk_id,
        "page": page,
        # 사업보고서/지배구조보고서: Cell 5c 키워드 탐지로 채워짐
        # 지속가능경영보고서: 목차 규칙이 없어 아직 빈 문자열 (팀 결정: 평가셋 이후 보강)
        "section_title": section_title,
        "content_type": content_type,
        "table_data": json.dumps(table_rows, ensure_ascii=False) if table_rows else "",
        "esg_area": "",  # TODO: 필요 시 규칙 기반 태깅
        "applicable_item_codes": json.dumps([], ensure_ascii=False),  # TODO
        "embedding_model": CONFIG["embedding_model"],
        "indexed_at": doc_indexed_at,  # 문서 단위 1회 생성값 재사용
        "source_file_hash": source_file_hash,
    }


### [Cell 5c] 키워드 기반 section_title 탐지 (사업보고서·지배구조보고서)

In [8]:
#
# 주의: 아래 키워드는 일반적인 법정 서식 기준의 "출발점"일 뿐, 회사·연도별로
# 소제목 문구나 넘버링 방식(로마자/아라비아숫자/장 표기 등)이 다를 수 있음.
# 정확한 문구 매칭 대신 "짧은 줄 안에 핵심 키워드가 포함되는지"만 보는 방식이라,
# 넘버링 형식이 달라도 잡히고, 매칭 실패 시 빈 문자열로 안전하게 degrade됨.
# 반드시 Cell Q2로 5~10개사 검증 후 키워드 목록을 조정할 것.
#
# 페이지 단위 granularity: 한 페이지 안에서 섹션이 바뀌면 그 페이지의 마지막
# 매칭을 채택 (청크 단위로 정밀하게 자르지는 않음 — 알려진 한계).
#
# [v4 패치] 리스트 순서 = 문서상 등장 순서라고 가정하고, 키워드 인덱스를 같이
# 반환하도록 변경. 예: 연결재무제표 주석의 "1. 회사의 개요"(각주 제목)처럼
# 이미 지나간 대분류 키워드가 뒤에서 다시 매칭되는 경우, 호출부(process_document)
# 에서 인덱스 역행을 감지해 무시한다. (검증 CSV에서 이 문제로 p.57~283이
# 전부 "회사의 개요"로 오염된 것 확인 — 재무제표 주석 안의 각주 제목이 원인)
SECTION_KEYWORDS = {
    "business_report": [
        "회사의 개요",
        "사업의 내용",
        "재무에 관한 사항",
        "이사의 경영진단",
        "회계감사인의 감사의견",
        "이사회 등 회사의 기관에 관한 사항",
        "주주에 관한 사항",
        "임원 및 직원 등에 관한 사항",
        "계열회사 등에 관한 사항",
        "이해관계자와의 거래내용",
    ],
    "governance_report": [
        "주주",
        "이사회",
        "감사기구",
        "이사 감사의 보수",
        "내부통제",
        "배당",
        "주주총회",
        "공시",
    ],
}


def _normalize(line: str) -> str:
    return line.replace("·", "").replace(" ", "").replace("/", "")


def detect_section_title(line: str, doc_type: str) -> tuple[str, int] | None:
    """짧고 키워드를 포함한 줄을 섹션 제목 후보로 판단.
    확신 없으면 None 반환 (호출부에서 빈 문자열로 처리 -> 안전).

    [v4 패치] 반환 타입을 str -> tuple[str, int]로 변경. int는
    SECTION_KEYWORDS 리스트 상의 순서 인덱스이며, 호출부에서 역행 판단에 사용.
    """
    keywords = SECTION_KEYWORDS.get(doc_type)
    if not keywords:
        return None

    stripped = line.strip()
    if not stripped or len(stripped) > 40:  # 제목치고 너무 길면 본문일 가능성 높음
        return None

    norm_line = _normalize(stripped)
    for idx, kw in enumerate(keywords):
        if _normalize(kw) in norm_line:
            return (kw, idx)
    return None


### [Cell Q2] (선택, 필수 실행 권장) section_title 탐지 정확도 수동 검증

In [9]:
# 5~10개사 정도 돌려서 페이지별로 섹션이 잘 감지되는지 눈으로 확인할 것.
# 엉뚱하게 잡히거나 안 잡히는 회사가 많으면 SECTION_KEYWORDS를 조정.
# [v4 패치] detect_section_title이 (제목, idx) 튜플을 반환하므로 미리보기에서도
# 역행 여부를 함께 표시 (실제 인덱싱 로직과 동일하게 역행은 무시하고 출력).
def preview_section_detection(file_path: str, doc_type: str):
    current_section = ""
    current_section_idx = -1
    with pdfplumber.open(file_path) as pdf:
        for page_number, page in enumerate(pdf.pages):
            text, _ = extract_page_content(page, doc_type)
            for line in text.split("\n"):
                detected = detect_section_title(line, doc_type)
                if not detected:
                    continue
                kw, idx = detected
                if idx < current_section_idx:
                    print(f"  p.{page_number + 1}: '{kw}' 매칭됐지만 역행이라 무시 "
                          f"(현재 섹션: '{current_section}')")
                    continue
                if kw != current_section:
                    current_section = kw
                    current_section_idx = idx
                    print(f"p.{page_number + 1}: '{current_section}' 섹션 시작 감지")


### [Cell Q] (선택) 표/텍스트 분리 품질 수동 확인용 진단 도구

In [10]:
# 본 파이프라인에 포함되지 않음. 실제 PDF 3~5개, 페이지 5~10장 정도
# 랜덤으로 뽑아 원본과 비교해볼 것 (GPT 지적 6번 대응).
# [v4 패치] extract_page_content가 doc_type을 받도록 바뀌어서 이 함수도 인자 추가
# (특히 지속가능경영보고서에서 표 fallback이 실제로 도는지 여기서 확인할 것).
def preview_page_extraction(file_path: str, page_number: int, doc_type: str = ""):
    with pdfplumber.open(file_path) as pdf:
        page = pdf.pages[page_number]
        original_text = page.extract_text() or ""
        clean_text, tables = extract_page_content(page, doc_type)
        print("=== 원본 전체 텍스트 (앞 500자) ===")
        print(original_text[:500])
        print(f"\n=== 표 제외 텍스트 (앞 500자) === (감지된 표 {len(tables)}개)")
        print(clean_text[:500])


In [11]:
# 현재 필터링된 manifest의 첫 번째 파일(SK하이닉스)로 테스트
test_file = manifest_df.iloc[0]["file_path"]
test_doc_type = manifest_df.iloc[0]["doc_type"]

# [1] Cell Q2 테스트: 섹션 감지가 잘 되는지 확인
print("=== [Cell Q2] 섹션 제목 감지 테스트 ===")
preview_section_detection(test_file, test_doc_type)

# [2] Cell Q 테스트: 10페이지의 표/본문 분리 확인 (원하는 페이지 번호 입력)
# [v4 패치] doc_type 전달 (지속가능경영보고서 표 fallback 확인용)
print("\n=== [Cell Q] 10페이지 본문/표 분리 테스트 ===")
preview_page_extraction(test_file, page_number=10, doc_type=test_doc_type)

# [3] (신규, 지속가능경영보고서만 해당) 표 fallback이 실제로 작동하는지 확인하고
# 싶으면, 지속가능경영보고서 파일로 아래처럼 별도 호출해볼 것:
# sus_row = manifest_df[manifest_df["doc_type"] == "sustainability_report"].iloc[0]
# preview_page_extraction(sus_row["file_path"], page_number=5,
#                          doc_type="sustainability_report")


=== [Cell Q2] 섹션 제목 감지 테스트 ===
p.6: '회사의 개요' 섹션 시작 감지
p.7: '사업의 내용' 섹션 시작 감지
p.48: '재무에 관한 사항' 섹션 시작 감지
  p.57: '회사의 개요' 매칭됐지만 역행이라 무시 (현재 섹션: '재무에 관한 사항')
  p.176: '회사의 개요' 매칭됐지만 역행이라 무시 (현재 섹션: '재무에 관한 사항')
p.295: '이사의 경영진단' 섹션 시작 감지
p.304: '회계감사인의 감사의견' 섹션 시작 감지
p.309: '이사회 등 회사의 기관에 관한 사항' 섹션 시작 감지
p.331: '주주에 관한 사항' 섹션 시작 감지
  p.337: '사업의 내용' 매칭됐지만 역행이라 무시 (현재 섹션: '주주에 관한 사항')
p.343: '임원 및 직원 등에 관한 사항' 섹션 시작 감지
p.365: '계열회사 등에 관한 사항' 섹션 시작 감지

=== [Cell Q] 10페이지 본문/표 분리 테스트 ===
=== 원본 전체 텍스트 (앞 500자) ===
투기성이 매우 높고 부도상태이거나 부도에 매우 근접해 있을 가능성이 높으며 원리금 회수 가능성이 어느 정
Ca
도 있음
C 채권의 최하 등급. 일반적으로 부도상태이고 원리금 회수 가능성이 거의 없음
※ Aa부터 Caa까지 각 일반 등급 분류에 1,2,3의 숫자를 부여함. 숫자 1은 해당 등급 분류 내에서 상위에 있음을, 숫자 2는 중위에, 숫자 3은 하위에 있음을
나타냄
아. 회사의 주권상장(또는 등록ㆍ지정) 및 특례상장에 관한 사항
주권상장 주권상장
특례상장 유형
(또는 등록ㆍ지정)현황 (또는 등록ㆍ지정)일자
유가증권시장 상장 1996년 12월 26일 해당사항 없음
2. 회사의 연혁
가. 회사의 본점소재지 및 그 변경
일 자 주소 비고
- 경기도 이천시 부발읍 경충대로 2091 공시대상 기간내 변경 없음
나. 경영진의 중요한 변동
(1) 공시대상기간 중 등기임원 변동 내역
선임 임기만료
변동일자 주총종류
신규 재선임 또는 해임
사내이사 이석희
[임기만료]

### [Cell 6] 문서 1건(PDF 1개) 처리 → Document 리스트 생성

In [12]:
# [v4 패치] 아래 두 헬퍼를 process_document보다 먼저 정의:
#   - strip_noise_lines: DART 뷰어 푸터("전자공시시스템 dart.fss.or.kr Page N")를
#     페이지 텍스트에서 제거. 검증 CSV에서 50자 미만 청크 134개 중 다수가 이 푸터였음.
#   - is_toc_chunk: 점선(".....")이 많이 포함된 목차성 청크 판별.
#     검증 CSV에서 목차 청크가 최대 5,388자까지 그대로 인덱싱되어 있었음.
import re

NOISE_LINE_PATTERN = re.compile(r"^전자공시시스템\s*dart\.fss\.or\.kr\s*Page\s*\d+\s*$")
TOC_DOT_PATTERN = re.compile(r"\.{10,}")


def strip_noise_lines(text: str) -> str:
    """DART 뷰어 푸터 줄 제거."""
    return "\n".join(
        ln for ln in text.split("\n") if not NOISE_LINE_PATTERN.match(ln.strip())
    )


def is_toc_chunk(chunk_text: str) -> bool:
    """점선 목차(.......) 덩어리 판별 (점 10개 이상 연속이 3군데 이상)."""
    return len(TOC_DOT_PATTERN.findall(chunk_text)) >= 3


def process_document(company_code: str, company_name: str, year: int,
                      doc_type: str, file_path: str,
                      source_file_hash: str, doc_indexed_at: str) -> list[Document]:

    doc_id = f"{company_code}_{year}_{doc_type}"
    chunks: list[Document] = []
    counter = 0
    current_section = ""      # 사업/지배구조보고서만 Cell 5c로 갱신됨 (페이지 단위 granularity)
    current_section_idx = -1  # [v4 패치] SECTION_KEYWORDS 상의 순서 인덱스, 역행 감지용

    with pdfplumber.open(file_path) as pdf:
        for page_number, page in enumerate(pdf.pages):
            # [v4 패치] doc_type 전달 -> 지속가능경영보고서는 표 fallback 적용됨
            text, tables = extract_page_content(page, doc_type)
            text = strip_noise_lines(text)  # [v4 패치] 푸터 줄 제거

            # 이 페이지에서 새 섹션 제목이 감지되면 갱신 (지속가능경영보고서는
            # SECTION_KEYWORDS에 키가 없어 detect_section_title이 항상 None 반환 -> 그대로 "")
            # [v4 패치] idx가 역행하면(이미 지나간 대분류가 다시 매칭되면) 무시.
            # 재무제표 주석의 "1. 회사의 개요" 같은 각주 제목 오탐 방지.
            for line in text.split("\n"):
                detected = detect_section_title(line, doc_type)
                if not detected:
                    continue
                kw, idx = detected
                if idx >= current_section_idx:
                    current_section = kw
                    current_section_idx = idx

            # --- 표 청크 (큰 표는 토큰 기준으로 여러 조각으로 분할) ---
            for table_data in tables:
                for piece in chunk_table(table_data, CONFIG["chunk_size_tokens"]):
                    counter += 1
                    chunk_id = f"{doc_id}_{counter:04d}"
                    meta = base_metadata(
                        company_code, company_name, year, doc_type, doc_id,
                        chunk_id, page_number + 1, "table", source_file_hash,
                        doc_indexed_at, table_rows=piece["rows"],
                        section_title=current_section,
                    )
                    meta["chunk_char_count"] = len(piece["markdown"])
                    chunks.append(Document(page_content=piece["markdown"], metadata=meta))

            # --- 텍스트 청크 (표 영역 제외된 텍스트만) ---
            if not text.strip():
                continue
            for chunk_text in splitter.split_text(text):
                # [v4 패치] 목차 덩어리 / 20자 미만 잔여 청크는 인덱싱하지 않음
                if is_toc_chunk(chunk_text) or len(chunk_text.strip()) < 20:
                    continue
                counter += 1
                chunk_id = f"{doc_id}_{counter:04d}"
                meta = base_metadata(
                    company_code, company_name, year, doc_type, doc_id,
                    chunk_id, page_number + 1, "text", source_file_hash,
                    doc_indexed_at, section_title=current_section,
                )
                meta["chunk_char_count"] = len(chunk_text)
                chunks.append(Document(page_content=chunk_text, metadata=meta))

    return chunks


In [13]:
# Cell 6 검증: SK하이닉스 첫 번째 문서 1건만 시험 청킹
sample = manifest_df.iloc[0]
sample_chunks = process_document(
    company_code=sample["company_code"],
    company_name=sample["company_name"],
    year=int(sample["year"]),
    doc_type=sample["doc_type"],
    file_path=sample["file_path"],
    source_file_hash="test",
    doc_indexed_at="test_time",
)

print(f"총 청크 수: {len(sample_chunks)}개")
print(f"첫 번째 청크 메타데이터:\n{sample_chunks[0].metadata}")
print(f"\n첫 번째 청크 내용:\n{sample_chunks[0].page_content[:200]}")

총 청크 수: 1404개
첫 번째 청크 메타데이터:
{'company_code': '000660', 'company_name': 'SK하이닉스', 'year': 2025, 'doc_type': 'business_report', 'doc_id': '000660_2025_business_report', 'chunk_id': '000660_2025_business_report_0001', 'page': 4, 'section_title': '', 'content_type': 'text', 'table_data': '', 'esg_area': '', 'applicable_item_codes': '[]', 'embedding_model': 'gemini-embedding-001', 'indexed_at': 'test_time', 'source_file_hash': 'test', 'chunk_char_count': 300}

첫 번째 청크 내용:
사 업 보 고 서
(제 77 기)
2024년 01월 01일 부터
사업연도
2024년 12월 31일 까지
금융위원회
한국거래소 귀중 2025년 3월 19일
제출대상법인 유형 : 주권상장법인
면제사유발생 : 해당사항 없음
회 사 명 : 에스케이하이닉스 주식회사
대 표 이 사 : 곽 노 정
본 점 소 재 지 : 경기도 이천시 부발읍 경충대로 2091
(전 화) 


### [Cell 7] manifest 순회 → 해시 기반 체크포인트 + 변경분만 재인덱싱

In [17]:
embeddings = GoogleGenerativeAIEmbeddings(model=CONFIG["embedding_model"])
vectorstore = Chroma(
    persist_directory=CONFIG["persist_dir"],
    embedding_function=embeddings,
)

if os.path.exists(CONFIG["checkpoint_log"]):
    with open(CONFIG["checkpoint_log"], encoding="utf-8") as f:
        checkpoint = json.load(f)  # {doc_id: {"source_file_hash":..., "indexed_at":...}}
else:
    checkpoint = {}

manifest = pd.read_csv(CONFIG["manifest_csv"], dtype={"company_code": str})

for _, row in manifest.iterrows():
    doc_id = f"{row['company_code']}_{row['year']}_{row['doc_type']}"
    file_path = row["file_path"]

    with open(file_path, "rb") as f:
        current_hash = hashlib.md5(f.read()).hexdigest()

    prior = checkpoint.get(doc_id)
    if prior and prior["source_file_hash"] == current_hash:
        print(f"스킵 (변경 없음): {doc_id}")
        continue

    if prior:
        print(f"PDF 변경 감지 → 기존 청크 삭제 후 재인덱싱: {doc_id}")
        vectorstore.delete(where={"doc_id": doc_id})

    print(f"처리 중: {doc_id}")
    try:
        doc_indexed_at = datetime.now().isoformat()
        chunks = process_document(
            company_code=row["company_code"],
            company_name=row["company_name"],
            year=int(row["year"]),
            doc_type=row["doc_type"],
            file_path=file_path,
            source_file_hash=current_hash,
            doc_indexed_at=doc_indexed_at,
        )
        if chunks:
            print(f"  청크 {len(chunks)}개, {CONFIG['embed_batch_size']}개씩 배치 임베딩 중...")
            add_documents_throttled(
                vectorstore, chunks,
                batch_size=CONFIG["embed_batch_size"],
                delay_sec=CONFIG["embed_batch_delay_sec"],
            )

        checkpoint[doc_id] = {
            "source_file_hash": current_hash,
            "indexed_at": doc_indexed_at,
        }
        with open(CONFIG["checkpoint_log"], "w", encoding="utf-8") as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)

    except Exception as e:
        print(f"실패 (건너뜀): {doc_id} — {e}")
        continue

print(f"\n완료: {len(checkpoint)}건 / 전체 {len(manifest)}건")

처리 중: 000660_2025_business_report
  청크 1404개, 100개씩 배치 임베딩 중...


처리 중: 000660_2025_governance_report
  청크 221개, 100개씩 배치 임베딩 중...
처리 중: 000660_2025_sustainability_report
  청크 379개, 100개씩 배치 임베딩 중...

완료: 3건 / 전체 3건


### [Cell 7b] (v4 신규) 코드 수정 후 재인덱싱 — 체크포인트 초기화
위 v4 패치(섹션/노이즈/표 fallback)를 반영한 뒤 다시 돌릴 때 필요.
PDF 파일 자체는 안 바뀌었으므로 Cell 7의 해시 체크가 그대로 통과해서 그냥 재실행하면 아무 일도 안 일어남.
아래 셀로 원하는 범위만 체크포인트를 지우고 기존 청크를 삭제한 뒤 Cell 7을 다시 실행할 것.

In [16]:
import shutil
import os
import json

# 1) 깨진 Drive 인덱스 폴더 통째로 삭제 (3건뿐이라 손실 없음)
if os.path.exists(CONFIG["persist_dir"]):
    shutil.rmtree(CONFIG["persist_dir"])
    print("깨진 Drive 인덱스 삭제 완료")

# 2) checkpoint도 같이 초기화 (인덱스가 사라졌으니 전부 재인덱싱 대상)
checkpoint = {}
with open(CONFIG["checkpoint_log"], "w", encoding="utf-8") as f:
    json.dump(checkpoint, f, ensure_ascii=False, indent=2)
print("checkpoint 초기화 완료 — Cell 7을 다시 실행할 것")

# 3) 앞으로는 Drive에 직접 안 쓰고 로컬(/content)에서 만든 뒤 마지막에 복사
LOCAL_PERSIST_DIR = "/content/chroma_db_local"
if os.path.exists(LOCAL_PERSIST_DIR):
    shutil.rmtree(LOCAL_PERSIST_DIR)
CONFIG["persist_dir"] = LOCAL_PERSIST_DIR  # Cell 7이 이 경로로 vectorstore 새로 만듦
print(f"persist_dir을 로컬 디스크로 변경: {LOCAL_PERSIST_DIR}")

깨진 Drive 인덱스 삭제 완료
checkpoint 초기화 완료 — Cell 7을 다시 실행할 것
persist_dir을 로컬 디스크로 변경: /content/chroma_db_local


In [19]:
# [검증 및 공유용] Chroma 인덱스 전체를 CSV로 export
raw = vectorstore.get(include=["documents", "metadatas"])

export_rows = []
for doc_id_chroma, content, meta in zip(raw["ids"], raw["documents"], raw["metadatas"]):
    row = dict(meta)
    row["content"] = content
    export_rows.append(row)

export_df = pd.DataFrame(export_rows)

front_cols = [
    "company_code", "company_name", "year", "doc_type",
    "page", "section_title", "content_type", "chunk_id", "content",
]
other_cols = [c for c in export_df.columns if c not in front_cols]
export_df = export_df[front_cols + other_cols]

# 파일명에 기업명을 붙여서 저장
export_path = "/content/drive/MyDrive/ESG프로젝트(개인)/데탐단/인덱스/3. RAG 인덱스/index_export_000660_SK하이닉스.csv"
export_df.to_csv(export_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {export_path} ({len(export_df)}행)")
export_df.head()

저장 완료: /content/drive/MyDrive/ESG프로젝트(개인)/데탐단/인덱스/3. RAG 인덱스/index_export_000660_SK하이닉스.csv (2004행)


,company_code,company_name,year,doc_type,page,section_title,content_type,chunk_id,content,applicable_item_codes,table_data,source_file_hash,embedding_model,doc_id,chunk_char_count,esg_area,indexed_at
0,000660,SK하이닉스,2025,business_report,4,,text,000660_2025_business_report_0001,사 업 보 고 서\n(제 77 기)\n2024년 01월 01일 부터\n사업연도\n2...,[],,f11f9c121fc80d110e53b92ba24fbad2,gemini-embedding-001,000660_2025_business_report,300,,2026-09-01T17:38:57.513553
1,000660,SK하이닉스,2025,business_report,6,회사의 개요,table,000660_2025_business_report_0002,| 구분 | 연결대상회사수 | | | | 주요 종속회사수 |\n|---|---...,[],"[[null, ""기초"", ""증가"", ""감소"", ""기말"", null], [""상장"", ...",f11f9c121fc80d110e53b92ba24fbad2,gemini-embedding-001,000660_2025_business_report,178,,2026-09-01T17:38:57.513553
2,000660,SK하이닉스,2025,business_report,6,회사의 개요,table,000660_2025_business_report_0003,| 구 분 | 자회사 | 사 유 |\n|---|---|---|\n| 신규 연결 | ...,[],"[[""신규\n연결"", ""SK hynix Semiconductor West Lafay...",f11f9c121fc80d110e53b92ba24fbad2,gemini-embedding-001,000660_2025_business_report,535,,2026-09-01T17:38:57.513553
3,000660,SK하이닉스,2025,business_report,6,회사의 개요,text,000660_2025_business_report_0004,I. 회사의 개요\n1. 회사의 개요\n가. 연결대상 종속회사 개황\n[연결대상 종...,[],,f11f9c121fc80d110e53b92ba24fbad2,gemini-embedding-001,000660_2025_business_report,384,,2026-09-01T17:38:57.513553
4,000660,SK하이닉스,2025,business_report,7,사업의 내용,table,000660_2025_business_report_0005,| 중소기업 해당 여부 | | 미해당 |\n|---|---|---|\n| | 벤...,[],"[["""", ""벤처기업 해당 여부"", ""미해당""], [""중견기업 해당 여부"", nul...",f11f9c121fc80d110e53b92ba24fbad2,gemini-embedding-001,000660_2025_business_report,85,,2026-09-01T17:38:57.513553


In [18]:
import shutil

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/ESG프로젝트(개인)/데탐단/인덱스/3. RAG 인덱스/chroma_db"
if os.path.exists(DRIVE_BACKUP_DIR):
    shutil.rmtree(DRIVE_BACKUP_DIR)
shutil.copytree(CONFIG["persist_dir"], DRIVE_BACKUP_DIR)
print(f"Drive로 백업 완료: {DRIVE_BACKUP_DIR}")

Drive로 백업 완료: /content/drive/MyDrive/ESG프로젝트(개인)/데탐단/인덱스/3. RAG 인덱스/chroma_db


### [Cell 8] 필터링 지원 하이브리드 리트리버

In [20]:
# 핵심 변경: BM25를 전체 코퍼스로 한 번에 만들지 않고, 검색 시점의 필터
# (company_code 등)로 좁혀진 부분집합에서만 만든다. 회사별로 캐싱해서
# 같은 회사를 다시 물어볼 때는 재구축 없이 재사용.
class FilteredHybridRetriever:
    def __init__(self, vectorstore, tokenize_func, k=10, weights=(0.5, 0.5)):
        self.vectorstore = vectorstore
        self.tokenize_func = tokenize_func
        self.k = k
        self.weights = weights
        self._bm25_cache: dict[tuple, BM25Retriever] = {}

    def _get_bm25_retriever(self, filter_dict):
        sig = tuple(sorted((filter_dict or {}).items()))
        if sig in self._bm25_cache:
            return self._bm25_cache[sig]

        raw = self.vectorstore.get(
            where=filter_dict if filter_dict else None,
            include=["documents", "metadatas"],
        )
        docs = [
            Document(page_content=d, metadata=m)
            for d, m in zip(raw["documents"], raw["metadatas"])
        ]
        if not docs:
            self._bm25_cache[sig] = None
            return None

        retriever = BM25Retriever.from_documents(docs, preprocess_func=self.tokenize_func)
        retriever.k = self.k
        self._bm25_cache[sig] = retriever
        return retriever

    def invoke(self, query: str, filter: dict | None = None, top_n: int | None = None):
        top_n = top_n or self.k

        dense_docs = self.vectorstore.similarity_search(query, k=self.k, filter=filter)

        bm25_retriever = self._get_bm25_retriever(filter)
        bm25_docs = bm25_retriever.invoke(query) if bm25_retriever else []

        # Reciprocal Rank Fusion
        scores: dict[str, float] = {}
        lookup: dict[str, Document] = {}
        for rank, doc in enumerate(dense_docs):
            key = doc.metadata["chunk_id"]
            scores[key] = scores.get(key, 0.0) + self.weights[0] * (1.0 / (RRF_K + rank))
            lookup[key] = doc
        for rank, doc in enumerate(bm25_docs):
            key = doc.metadata["chunk_id"]
            scores[key] = scores.get(key, 0.0) + self.weights[1] * (1.0 / (RRF_K + rank))
            lookup[key] = doc

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [lookup[k] for k, _ in ranked[:top_n]]

    def clear_cache(self):
        """회사 데이터를 재인덱싱한 뒤에는 캐시가 stale할 수 있으니 호출."""
        self._bm25_cache.clear()


hybrid_retriever = FilteredHybridRetriever(
    vectorstore=vectorstore,
    tokenize_func=kiwi_tokenize,
    k=CONFIG["retrieval_k"],
    weights=CONFIG["hybrid_weights"],
)

### [Cell 9] 동작 확인 — 반드시 실행해볼 것

In [21]:
# (1) dense 메타데이터 사전 필터 확인
test_filtered = vectorstore.similarity_search(
    "사외이사 비율", k=5, filter={"company_code": "000660"},
)
for d in test_filtered:
    assert d.metadata["company_code"] == "000660", "필터링 실패 — 다른 회사 청크 혼입"
print(f"Dense 필터 테스트 통과: {len(test_filtered)}건")

# (2) 하이브리드 검색도 필터가 실제로 걸리는지 확인 (v2까지 안 되던 부분)
hybrid_results = hybrid_retriever.invoke(
    "이사회 산하 위원회 구성 현황", filter={"company_code": "000660"},
)
for d in hybrid_results:
    assert d.metadata["company_code"] == "000660", "하이브리드 필터링 실패"
print(f"하이브리드 필터 테스트 통과: {len(hybrid_results)}건")
for r in hybrid_results[:3]:
    print(r.metadata["company_code"], r.metadata["doc_type"],
          r.metadata["page"], "|", r.page_content[:80])


# ==============================================================================

Dense 필터 테스트 통과: 5건
하이브리드 필터 테스트 통과: 10건
000660 governance_report 45 | 시
행
Y(O)
여
부
당사는 이사회의 효율적 운영과 전문성을 위하여 이사회 산하에 감사위원회, 사외이사후보추천위원회, 지속경영위원회, 인사·보
000660 business_report 316 | (주1) 2024년 4월 22일 감사위원회에서 한애라 사외이사를 감사위원회 위원장으로 신규 선임하였습니다.
(주2) 2021년 3월 30일 보상
000660 governance_report 46 | 명
문
규
Y(O)
정
여
부
당사는 이사회 내 위원회의 설치 목적, 조직, 운영, 권한 및 책임에 대해서는 각각의 위원회별 규정을 통해 서면으


### [Cell R] 세션 재시작 시 — 이 셀만 실행하면 됨 (Cell 1~4는 재실행 필요)

In [ ]:
# v2와 달리 전역 BM25를 pickle로 저장/복원하지 않으므로 로드가 훨씬 가벼움
# ==============================================================================
def load_existing_index():
    _embeddings = GoogleGenerativeAIEmbeddings(model=CONFIG["embedding_model"])
    _vectorstore = Chroma(
        persist_directory=CONFIG["persist_dir"],
        embedding_function=_embeddings,
    )
    _hybrid_retriever = FilteredHybridRetriever(
        vectorstore=_vectorstore,
        tokenize_func=kiwi_tokenize,
        k=CONFIG["retrieval_k"],
        weights=CONFIG["hybrid_weights"],
    )
    print("기존 인덱스 로드 완료 (BM25는 회사별로 필요할 때 자동 생성됨)")
    return _vectorstore, _hybrid_retriever


# 사용 예: vectorstore, hybrid_retriever = load_existing_index()